# AutoSort Training Pipeline

Main steps:
1. Threshold detection
2. Training data preparation
3. Model training


In [2]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre

from pathlib import Path
from utils_clean import (
    prepare_training_data,
    train_autosort_model
)


In [4]:
# Load data
recording_path = '/media/ubuntu/sda/data/mouse11/ns4/natural_image/mouse11_021722_natural_image_001.ns4'
spike_inf_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/other_mouse/mouse11_ni_sorter_output/021722/spike_inf.tsv"
neuron_inf_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/other_mouse/mouse11_ni_sorter_output/021722/neuron_inf.pkl"

# Load GT data
spike_inf = pd.read_csv(spike_inf_path, sep='\t', index_col=0)
with open(neuron_inf_path, 'rb') as f:
    neuron_inf = pickle.load(f)

# Load and preprocess recording
recording_raw = se.read_blackrock(file_path=recording_path)
recording_recorded = recording_raw.remove_channels(["98", '31', '32'])
recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
recording_f = spre.common_reference(recording_f, reference="global", operator="median")

print(f"Recording loaded successfully")
print(f"Sampling rate: {recording_f.get_sampling_frequency()} Hz")
print(f"Number of channels: {recording_f.get_num_channels()}")


Recording loaded successfully
Sampling rate: 10000.0 Hz
Number of channels: 30


## Step 1: Threshold Detection + Training Data Preparation


In [5]:
# Set parameters
save_dir = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/other_mouse/mouse11_ni_sorter_output/autosort_input/"
duration_seconds = 200  # Processing duration (seconds)

# Extract all unique tract_channels from neuron_inf for threshold detection on these channels only
valid_channels = sorted(neuron_inf['tract_channel'].unique().tolist())
print(f"Number of valid channels extracted from neuron_inf: {len(valid_channels)}")
print(f"Valid channels list: {valid_channels}")

# Detection parameters (consistent with AutoSort default values)
detection_params = {
    'thr_min': 3.5,
    'thr_max': 30,
    'distance': 3,
    'ch_max_simul_firing': 5,
    'wlen': 5,
    'prominence': 10,
}

# Waveform window parameters
window_params = {
    'left_sample': 10,
    'right_sample': 20,
}

# Prepare training data (includes threshold detection, GT matching, waveform extraction, data saving)
train_data_dir = prepare_training_data(
    recording_f=recording_f,
    spike_inf=spike_inf,
    neuron_inf=neuron_inf,
    save_dir=save_dir,
    duration_seconds=duration_seconds,
    valid_channels=valid_channels,  # Pass valid_channels parameter to detect only on valid channels
    **detection_params,
    **window_params
)


Number of valid channels extracted from neuron_inf: 24
Valid channels list: [0, 2, 3, 4, 5, 6, 7, 9, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 26, 27, 28, 29]
### 1. Threshold Detection
Sampling rate: 10000.0 Hz, Number of channels: 30
Recording total length: 36957913 samples (3695.79 seconds)
Will process first 2000000 samples (200.00 seconds)
Number of valid channels: 24
Valid channels list: [0, 2, 3, 4, 5, 6, 7, 9, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 26, 27, 28, 29]
Data shape: (2000000, 30)
Building detect_array...
Number of detected spikes: 420097

### 2. Load Ground Truth and Match
Building gt_array...
GT spike count: 83245
---spike detection rate: 0.9739
Number of matched spikes: 81069
Number of unmatched spikes: 339028

### 3. Extract Waveforms


Extracting waveforms: 100%|██████████| 30/30 [00:09<00:00,  3.25it/s]


Waveform extraction completed!
waveform shape: (420095, 30, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/other_mouse/mouse11_ni_sorter_output/autosort_input/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/other_mouse/mouse11_ni_sorter_output/autosort_input/train_data
Data statistics:
  - Total spike count: 420095
  - Number of channels: 30
  - Window length: 30
  - Number of unique units: 25
  - Noise spike count: 339027
  - Valid spike count: 81068


## Step 2: Model Training


In [6]:
# Set training parameters
base_model_save_dir = save_dir + "model_save/"
n_channels = recording_f.get_num_channels()

training_params = {
    'epochs': 20,
    'batch_size': 512,
    'left_sample': 10,
    'right_sample': 20,
    'early_stopping': True,  # Enable early stopping
    'patience': 5,  # Stop if accuracy doesn't improve for 5 consecutive epochs
    'min_delta': 0.0,  # Minimum change
}

# Repeat training 5 times
n_runs = 5
all_models = []
all_logs = []

for run_id in range(1, n_runs + 1):
    print(f"\n{'='*60}")
    print(f"Starting training run {run_id}/{n_runs}")
    print(f"{'='*60}")
    
    # Create independent save directory for each training run
    model_save_dir = base_model_save_dir + f"run_{run_id}/"
    
    # Train model
    autosort_model, training_log = train_autosort_model(
        train_data_dir=train_data_dir,
        model_save_dir=model_save_dir,
        n_channels=n_channels,
        **training_params
    )
    
    all_models.append(autosort_model)
    all_logs.append(training_log)
    
    print(f"\nTraining run {run_id} completed!")
    print(f"Model save directory: {model_save_dir}")

print(f"\n{'='*60}")
print(f"All {n_runs} training runs completed!")
print(f"{'='*60}")



Starting training run 1/5
Using device: cuda
Create dataset...
Dataset loaded:
  - Total samples: 420095
  - Number of channels: 30
  - Window length: 30
  - Number of unique units: 25
  - Noise samples: 339027.0
  - Non-noise samples: 81068.0
Model parameters:
  - Number of channels: 30
  - Window length: 30
  - Number of units: 25
  - Input dimension: 930
Unit ID list saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/other_mouse/mouse11_ni_sorter_output/autosort_input/model_save/run_1/keep_id.pkl

Dataset split:
  - Training set: 336076 samples
  - Validation set: 84019 samples

Starting training (total 20 epochs)...
Early stopping enabled: patience=5, min_delta=0.0
epoch : 1/20


Training: 100%|██████████| 657/657 [00:04<00:00, 132.33it/s]


epoch : 1/20, detection loss = 336.002335, classification loss = 623.352117


Validation: 100%|██████████| 165/165 [00:00<00:00, 199.89it/s]


epoch : 1/20, val detection loss = 278.183372, classification loss = 333.269866
Validation Loss Decreased(inf--->611.453238)
Validation Accuracy Decreased(inf--->0.878266) 	 Saving The Model (Best Acc Epoch)
epoch : 2/20


...

epoch : 5/20, val detection loss = 295.157035, classification loss = 79.161884
epoch : 6/20


Training: 100%|██████████| 657/657 [00:04<00:00, 140.14it/s]


epoch : 6/20, detection loss = 91.581389, classification loss = 36.221742


Validation: 100%|██████████| 165/165 [00:00<00:00, 228.60it/s]

epoch : 6/20, val detection loss = 315.430734, classification loss = 80.168139

Early stopping triggered: 5 consecutive epochs without improvement
Best accuracy: 0.872517 (Epoch 1)

Training completed! Training log saved to: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/other_mouse/mouse11_ni_sorter_output/autosort_input/model_save/run_5/training_log.csv
Best validation accuracy: 0.872517 (Epoch 1)

Training run 5 completed!
Model save directory: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/other_mouse/mouse11_ni_sorter_output/autosort_input/model_save/run_5/

All 5 training runs completed!
